In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("olympics_capstone.db")

In [5]:
query = """

SELECT *
FROM athlete_events
LIMIT 5;

"""

pd.read_sql_query(query, conn)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,No Medal
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,No Medal
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,No Medal
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,No Medal


In [6]:
query = """

SELECT *
FROM noc_regions
LIMIT 5;

"""

pd.read_sql_query(query, conn)

,NOC,region,notes
0,AFG,Afghanistan,
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,
3,ALG,Algeria,
4,AND,Andorra,


In [10]:
query = """

--Before deciding what relationships we want to explore we need to understand and know where
--the missing values are. This as a reminder of the missing values.

SELECT
    COUNT(*) AS total_records
    ,SUM (CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS missing_age_value
    ,ROUND(100.0*(SUM (CASE WHEN age IS NULL THEN 1 ELSE 0 END))/COUNT(*),2) AS percentage_of_missing_age_value
    ,SUM (CASE WHEN height IS NULL THEN 1 ELSE 0 END) AS missing_height_value
    ,ROUND(100.0*(SUM (CASE WHEN height IS NULL THEN 1 ELSE 0 END))/COUNT(*),2) AS percentage_of_missing_height_value
    ,SUM (CASE WHEN weight IS NULL THEN 1 ELSE 0 END) AS missing_weight_value
    ,ROUND(100.0*(SUM (CASE WHEN weight IS NULL THEN 1 ELSE 0 END))/COUNT(*),2) AS percentage_of_missing_weight_value
FROM athlete_events


"""

pd.read_sql_query(query, conn)

,total_records,missing_age_value,percentage_of_missing_age_value,missing_height_value,percentage_of_missing_height_value,missing_weight_value,percentage_of_missing_weight_value
0,271116,9474,3.49,60171,22.19,62875,23.19


In [13]:
query = """

SELECT
    year
    ,COUNT(*) AS total_records
    ,SUM (CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS missing_age_value
    ,ROUND(100.0*(SUM (CASE WHEN age IS NULL THEN 1 ELSE 0 END))/COUNT(*),2) AS percentage_of_missing_age_value
    ,SUM (CASE WHEN height IS NULL THEN 1 ELSE 0 END) AS missing_height_value
    ,ROUND(100.0*(SUM (CASE WHEN height IS NULL THEN 1 ELSE 0 END))/COUNT(*),2) AS percentage_of_missing_height_value
    ,SUM (CASE WHEN weight IS NULL THEN 1 ELSE 0 END) AS missing_weight_value
    ,ROUND(100.0*(SUM (CASE WHEN weight IS NULL THEN 1 ELSE 0 END))/COUNT(*),2) AS percentage_of_missing_weight_value
FROM athlete_events
GROUP BY Year
LIMIT 10



"""

pd.read_sql_query(query, conn)

,Year,total_records,missing_age_value,percentage_of_missing_age_value,missing_height_value,percentage_of_missing_height_value,missing_weight_value,percentage_of_missing_weight_value
0,1896,380,163,42.89,334,87.89,331,87.11
1,1900,1936,790,40.81,1820,94.01,1857,95.92
2,1904,1301,274,21.06,1088,83.63,1154,88.70
3,1906,1733,743,42.87,1476,85.17,1528,88.17
4,1908,3101,649,20.93,2626,84.68,2618,84.42
5,1912,4040,156,3.86,3319,82.15,3444,85.25
6,1920,4292,845,19.69,3525,82.13,3821,89.03
7,1924,5693,1142,20.06,4719,82.89,5003,87.88
8,1928,5574,963,17.28,4599,82.51,4856,87.12
9,1932,3321,330,9.94,2108,63.47,2771,83.44


In [18]:
query = """ 

--Of all missing height/weight/age values in the whole dataset, what % comes from each year?
--We also add a cummulative column for each field
--This is useful to see where the missing data is concentrated in

WITH missing_values_per_year AS (
SELECT
    year
    ,SUM (CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS missing_age
    ,SUM (CASE WHEN height IS NULL THEN 1 ELSE 0 END) AS missing_height
    ,SUM (CASE WHEN weight IS NULL THEN 1 ELSE 0 END) AS missing_weight
FROM athlete_events
GROUP BY Year

),

total_missing_values AS (
SELECT
    SUM (CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS missing_age
    ,SUM (CASE WHEN height IS NULL THEN 1 ELSE 0 END) AS missing_height
    ,SUM (CASE WHEN weight IS NULL THEN 1 ELSE 0 END) AS missing_weight
FROM athlete_events
)

SELECT
    m.year
    ,m.missing_age
    ,ROUND(100.0*m.missing_age / t.missing_age,2) AS percentage_of_all_missing_age_value
    ,ROUND(
        100.0 * SUM(m.missing_age) OVER (ORDER BY m.Year) / t.missing_age,
        2
    ) AS cumulative_pct_missing_age
    ,m.missing_height
    ,ROUND(100.0*m.missing_height / t.missing_height,2) AS percentage_of_all_missing_height_value
    ,ROUND(
        100.0 * SUM(m.missing_height) OVER (ORDER BY m.Year) / t.missing_height,
        2
    ) AS cumulative_pct_missing_height
    ,m.missing_weight
    ,ROUND(100.0*m.missing_weight / t.missing_weight,2) AS percentage_of_all_missing_weight_value
    ,ROUND(
        100.0 * SUM(m.missing_weight) OVER (ORDER BY m.Year) / t.missing_weight,
        2
    ) AS cumulative_pct_missing_weight
FROM missing_values_per_year m 
CROSS JOIN total_missing_values t
GROUP BY m.year

"""

pd.read_sql_query(query, conn)

,year,missing_age,percentage_of_all_missing_age_value,cumulative_pct_missing_age,missing_height,percentage_of_all_missing_height_value,cumulative_pct_missing_height,missing_weight,percentage_of_all_missing_weight_value,cumulative_pct_missing_weight
0,1896,163,1.72,1.72,334,0.56,0.56,331,0.53,0.53
1,1900,790,8.34,10.06,1820,3.02,3.58,1857,2.95,3.48
2,1904,274,2.89,12.95,1088,1.81,5.39,1154,1.84,5.32
3,1906,743,7.84,20.79,1476,2.45,7.84,1528,2.43,7.75
4,1908,649,6.85,27.64,2626,4.36,12.21,2618,4.16,11.91
5,1912,156,1.65,29.29,3319,5.52,17.72,3444,5.48,17.39
6,1920,845,8.92,38.21,3525,5.86,23.58,3821,6.08,23.46
7,1924,1142,12.05,50.26,4719,7.84,31.42,5003,7.96,31.42
8,1928,963,10.16,60.43,4599,7.64,39.07,4856,7.72,39.14
9,1932,330,3.48,63.91,2108,3.50,42.57,2771,4.41,43.55


In [20]:
query = """

--We want to determine which time frame we should use to analyze physical attributes
--where the missing data is less than 10%

SELECT
    year
    ,COUNT (*) AS total_records
    ,ROUND(100.0 * SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_age_pct
    ,ROUND(100.0 * SUM(CASE WHEN Height IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_height_pct
    ,ROUND(100.0 * SUM(CASE WHEN Weight IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_weight_pct
FROM athlete_events
GROUP BY Year
HAVING 
    missing_age_pct <= 10
    AND missing_height_pct <= 10
    AND missing_weight_pct <= 10
ORDER BY year

"""

pd.read_sql_query(query, conn)


,Year,total_records,missing_age_pct,missing_height_pct,missing_weight_pct
0,1964,9480,0.59,7.18,7.47
1,1968,10479,1.13,1.46,1.61
2,1972,11959,0.80,2.52,3.25
3,1976,10502,0.50,8.34,8.76
4,1980,8937,2.09,6.58,6.67
5,1984,11588,1.86,5.16,5.20
6,1988,14676,0.75,6.36,6.32
7,1994,3160,0.06,5.92,5.98
8,1998,3605,0.06,2.33,2.39
9,2000,13821,0.01,0.89,0.91


In [28]:
query = """

--Now we will begin exploring relationships, starting with country 
--participation vs medal success
--Our main variables will be participation volume, and medal success
--Here medal success is either medal volumn and medal rate
--We start with a descriptive table to gain intuition

SELECT 
    n.region
    ,COUNT(*) AS total_participations
    ,SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) AS total_medals
    ,ROUND(100.0*(SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END))/COUNT(*),2) AS medal_rate
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY n.region
ORDER BY total_participations DESC
LIMIT 20
    
"""

pd.read_sql_query(query, conn)

,region,total_participations,total_medals,medal_rate
0,USA,18853,5637,29.90
1,Germany,15883,3756,23.65
2,France,12758,1777,13.93
3,UK,12256,2068,16.87
4,Russia,11692,3947,33.76
5,Italy,10715,1637,15.28
6,Canada,9734,1352,13.89
7,Japan,8444,913,10.81
8,Sweden,8339,1536,18.42
9,Australia,7724,1349,17.47


In [51]:
query = """

--Now we want to explore the actual correlation between total participations and 'medal success'
--Total participation will be different from unique athletes, because medals are recorded at the athlete-event level
--Because SQLite 3 here has no correlation function, we switch back to pandas to do a correlation table
--Participation vs total medals: 0.9048
--Participation vs medal rate: 0.6133

WITH descriptive_table AS 
(
 SELECT 
    n.region
    ,COUNT(*) AS total_participations
    ,SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) AS total_medals
    ,ROUND(100.0*(SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END))/COUNT(*),2) AS medal_rate
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY n.region
ORDER BY total_participations DESC  
)

SELECT*
FROM descriptive_table

"""

descriptive_table = pd.read_sql_query(query, conn)

descriptive_table[["total_participations", "total_medals", "medal_rate"]].corr()




,total_participations,total_medals,medal_rate
total_participations,1.000000,0.926456,0.740971
total_medals,0.926456,1.000000,0.719716
medal_rate,0.740971,0.719716,1.000000


In [50]:
query = """

-- Now we want to know Do countries that concentrate more of their participation in a specific sport tend to achieve better medal success in that sport?
--Sport specialization will be country_sport_participation_share = country’s participation records in a sport / country’s total participation records

WITH country_sport_participation AS
(
SELECT
    a.noc
    ,n.region AS country
    ,a.sport AS sport
    ,COUNT(*) total_sport_participation
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY country, a.sport
HAVING total_sport_participation > 0
),

country_total_participation AS
(
SELECT
    a.noc
    ,n.region AS country
    ,COUNT(*) AS total_participation
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY country
),

sport_specialization_table AS
(
SELECT 
    csp.country AS country
    ,csp.sport as sport
    ,csp.total_sport_participation
    ,ctp.total_participation
    ,ROUND(100.0*csp.total_sport_participation / ctp.total_participation,2) AS sport_specialization
FROM country_sport_participation csp JOIN country_total_participation ctp
    ON csp.noc = ctp.noc
GROUP BY csp.country, csp.sport
),

medals_table AS 
(
SELECT
    n.region AS country
    ,a.sport AS sport
    ,SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) AS total_medals
    ,ROUND(100.0*(SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END))/COUNT(*),2) AS medal_rate
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY country, sport
)

SELECT
    sst.country
    ,sst.sport
    ,sst.sport_specialization
    ,mt.total_medals
    ,mt.medal_rate
FROM sport_specialization_table sst JOIN medals_table mt
    ON sst.country = mt.country AND sst.sport =mt.sport
WHERE sst.total_sport_participation >= 50
ORDER BY sst.country, sst.sport_specialization DESC

    
"""

pd.read_sql_query(query, conn)

,country,sport,sport_specialization,total_medals,medal_rate
0,Algeria,Athletics,25.23,9,6.47
1,Algeria,Boxing,12.52,6,8.70
2,Algeria,Handball,10.71,0,0.00
3,Algeria,Judo,9.44,2,3.85
4,Andorra,Alpine Skiing,62.13,0,0.00
...,...,...,...,...,...
1081,"Virgin Islands, US",Swimming,27.55,0,0.00
1082,"Virgin Islands, US",Athletics,20.75,0,0.00
1083,Zambia,Athletics,38.25,1,1.43
1084,Zambia,Boxing,27.87,1,1.96


In [53]:
query = """

--Now will once again explore the correlation between these two variables
--sport_specialization vs total medals: -0.01
--sport_specialization vs medal rate: -0.18

WITH country_sport_participation AS
(
SELECT
    a.noc
    ,n.region AS country
    ,a.sport AS sport
    ,COUNT(*) total_sport_participation
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY country, a.sport
HAVING total_sport_participation > 0
),

country_total_participation AS
(
SELECT
    a.noc
    ,n.region AS country
    ,COUNT(*) AS total_participation
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY country
),

sport_specialization_table AS
(
SELECT 
    csp.noc
    ,csp.country AS country
    ,csp.sport as sport
    ,csp.total_sport_participation
    ,ctp.total_participation
    ,ROUND(100.0*csp.total_sport_participation / ctp.total_participation,2) AS sport_specialization
FROM country_sport_participation csp JOIN country_total_participation ctp
    ON csp.noc = ctp.noc
),

medals_table AS 
(
SELECT
    a.noc
    ,n.region AS country
    ,a.sport AS sport
    ,SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) AS total_medals
    ,ROUND(100.0*(SUM (CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END))/COUNT(*),2) AS medal_rate
FROM noc_regions n LEFT JOIN athlete_events a
    ON n.noc = a.noc
GROUP BY country, sport
),

descriptive_table AS 
(
SELECT
    sst.country
    ,sst.sport
    ,sst.sport_specialization
    ,mt.total_medals
    ,mt.medal_rate
FROM sport_specialization_table sst JOIN medals_table mt
    ON sst.noc = mt.noc AND sst.sport =mt.sport
WHERE sst.total_sport_participation >= 50
ORDER BY sst.country, sst.sport_specialization DESC
)

SELECT*
FROM descriptive_table

"""

descriptive_table = pd.read_sql_query(query, conn)

descriptive_table[["sport_specialization", "total_medals", "medal_rate"]].corr()


,sport_specialization,total_medals,medal_rate
sport_specialization,1.000000,-0.017562,-0.185518
total_medals,-0.017562,1.000000,0.552618
medal_rate,-0.185518,0.552618,1.000000


In [69]:
query = """

--We will now see if there is a relationship between physical attributes and medal status
--We use the timeframe where the missing data is less than 10%

WITH no_missing_data AS
(
SELECT
    year
    ,COUNT (*) AS total_records
    ,ROUND(100.0 * SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_age_pct
    ,ROUND(100.0 * SUM(CASE WHEN Height IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_height_pct
    ,ROUND(100.0 * SUM(CASE WHEN Weight IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_weight_pct
FROM athlete_events
GROUP BY Year
HAVING 
    missing_age_pct <= 10
    AND missing_height_pct <= 10
    AND missing_weight_pct <= 10
),

valid_dataset AS 
(
SELECT
    a.*
FROM no_missing_data md INNER JOIN athlete_events a
    ON md.year = a.year
),

medalists AS
(
SELECT 
    sport
    ,COUNT(*) AS medalist_records
    ,AVG(age) AS avg_age
    ,AVG(height) AS avg_height
    ,AVG(weight) AS avg_weight
FROM valid_dataset
WHERE medal IN ('Gold','Silver','Bronze')
GROUP BY sport
),

non_medalists AS
(
SELECT 
    sport
    ,COUNT(*) AS non_medalist_records
    ,AVG(age) AS avg_age
    ,AVG(height) AS avg_height
    ,AVG(weight) AS avg_weight
FROM valid_dataset
WHERE medal NOT IN ('Gold','Silver','Bronze')
GROUP BY sport
)

SELECT
    m.sport
    ,m.medalist_records 
    ,nm.non_medalist_records
    ,m.avg_age AS m_avg_age
    ,nm.avg_age AS nm_avg_age
    ,ROUND((m.avg_age-nm.avg_age),2) AS age_gap
    ,m.avg_height AS m_avg_height
    ,nm.avg_height AS nm_avg_height
    ,ROUND((m.avg_height-nm.avg_height),2) AS height_gap
    ,m.avg_weight AS m_avg_weight
    ,nm.avg_weight AS nm_avg_weight
    ,ROUND((m.avg_weight-nm.avg_weight),2) AS weight_gap
FROM medalists m JOIN non_medalists nm
    ON m.sport = nm.sport
ORDER BY weight_gap DESC
LIMIT 5

"""

pd.read_sql_query(query, conn)
    

,sport,medalist_records,non_medalist_records,m_avg_age,nm_avg_age,age_gap,m_avg_height,nm_avg_height,height_gap,m_avg_weight,nm_avg_weight,weight_gap
0,Golf,6,114,31.166667,29.438596,1.73,179.166667,173.903509,5.26,80.500000,70.836538,9.66
1,Cycling,678,5666,25.470501,25.267854,0.20,177.986667,175.952347,2.03,73.469630,69.682436,3.79
2,Softball,134,224,27.029851,26.026786,1.00,170.671642,168.959821,1.71,70.119403,66.625000,3.49
3,Swimming,1975,14152,21.072405,20.529621,0.54,181.494877,178.167107,3.33,73.425205,70.054966,3.37
4,Rugby Sevens,74,225,25.337838,26.280000,-0.94,176.594595,174.955157,1.64,81.256757,78.177778,3.08


In [72]:
query = """

--Now we will dive deeper and analyze the top 3 most popular sports, and see if there is a correlation between physical attributes and medal status

SELECT
    sport
    ,count(*)
FROM athlete_events
GROUP BY sport
ORDER BY count(*) DESC
LIMIT 5

"""

pd.read_sql_query(query, conn)


,Sport,count(*)
0,Athletics,38624
1,Gymnastics,26707
2,Swimming,23195
3,Shooting,11448
4,Cycling,10859


In [76]:
query = """

--We start with Atheltics 
--height vs medal_status: 0.052
--weight vs medal_status: 0.052
--age vs medal_status: -0.011

WITH descriptive_table AS (
SELECT
    age
    ,height
    ,weight
    ,CASE 
        WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END AS medal_status
FROM athlete_events
WHERE sport = 'Athletics'
)

SELECT*
FROM descriptive_table

"""

descriptive_table = pd.read_sql_query(query, conn)

descriptive_table[["age", "height", "weight",'medal_status']].corr()


,age,height,weight,medal_status
age,1.000000,0.012933,0.066148,-0.010967
height,0.012933,1.000000,0.752957,0.052025
weight,0.066148,0.752957,1.000000,0.052093
medal_status,-0.010967,0.052025,0.052093,1.000000


In [77]:
query = """

--Now with Gymnastics
--height vs medal_status: -0.043
--weight vs medal_status: -0.049
--age vs medal_status: 0.042

WITH descriptive_table AS (
SELECT
    age
    ,height
    ,weight
    ,CASE 
        WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END AS medal_status
FROM athlete_events
WHERE sport = 'Gymnastics'
)

SELECT*
FROM descriptive_table

"""

descriptive_table = pd.read_sql_query(query, conn)

descriptive_table[["age", "height", "weight",'medal_status']].corr()

,age,height,weight,medal_status
age,1.000000,0.471709,0.578586,0.042028
height,0.471709,1.000000,0.868707,-0.043173
weight,0.578586,0.868707,1.000000,-0.048607
medal_status,0.042028,-0.043173,-0.048607,1.000000


In [78]:
query = """

--Now with Swimming
--height vs medal_status: 0.094
--weight vs medal_status: 0.092
--age vs medal_status: 0.037

WITH descriptive_table AS (
SELECT
    age
    ,height
    ,weight
    ,CASE 
        WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END AS medal_status
FROM athlete_events
WHERE sport = 'Swimming'
)

SELECT*
FROM descriptive_table

"""

descriptive_table = pd.read_sql_query(query, conn)

descriptive_table[["age", "height", "weight",'medal_status']].corr()

,age,height,weight,medal_status
age,1.000000,0.349405,0.359694,0.037443
height,0.349405,1.000000,0.864016,0.093812
weight,0.359694,0.864016,1.000000,0.091754
medal_status,0.037443,0.093812,0.091754,1.000000
